# Дообучение YandexGPT-5-Lite-8B-pretrain

Ноутбук подготавливает LoRA/SFT-обучение для задачи преобразования русских аналитических запросов в JSON-фильтры API.


## Что нужно до запуска

- GPU с CUDA и желательно 16+ GB VRAM но у меня 8 GB и работает
- `HF_TOKEN` в корневом `.env`
- датасет уже лежит в `training/llm_sft/`


In [ ]:
!pip install -q -U "transformers>=4.57.0" "accelerate>=1.7.0" "peft>=0.18.0" "trl>=0.23.0" "bitsandbytes>=0.48.0" datasets sentencepiece huggingface_hub


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import torch
from huggingface_hub import login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer


In [ ]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_env_file(repo_root / '.env')
hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN не найден в .env')

login(token=hf_token, add_to_git_credential=False)
print('HF token loaded successfully')


In [ ]:
MODEL_NAME = 'yandex/YandexGPT-5-Lite-8B-pretrain'
TRAIN_PATH = repo_root / 'training' / 'llm_sft' / 'budget_query_sft_train.jsonl'
VAL_PATH = repo_root / 'training' / 'llm_sft' / 'budget_query_sft_val.jsonl'
DATASET_CACHE_DIR = repo_root / '.cache' / 'hf_datasets'
OUTPUT_DIR = repo_root / 'artifacts' / 'yandexgpt5-lite-budget-query-lora'

DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MAX_SEQ_LENGTH = 1024
print(TRAIN_PATH)
print(VAL_PATH)
print(DATASET_CACHE_DIR)


In [ ]:
from training.llm_sft.dataset_loader import load_budget_query_sft_dataset

dataset = load_budget_query_sft_dataset(cache_dir=DATASET_CACHE_DIR)
dataset


In [ ]:
sample = dataset['train'][0]
print(sample['text_query'])
print(sample['prompt'][:700])
print(sample['completion'])


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Для дообучения нужен GPU с CUDA. Для T4 этого достаточно.')

major, minor = torch.cuda.get_device_capability(0)
supports_bf16 = major >= 8
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
MAX_SEQ_LENGTH = 1536 if supports_bf16 else 1024
gradient_accumulation_steps = 16 if supports_bf16 else 32
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('GPU:', torch.cuda.get_device_name(0))
print('CUDA capability:', f'{major}.{minor}')
print('bf16 supported:', supports_bf16)
print('MAX_SEQ_LENGTH:', MAX_SEQ_LENGTH)

from huggingface_hub import snapshot_download
local_model_dir = snapshot_download(
    repo_id=MODEL_NAME,
    token=hf_token,
    cache_dir=str(repo_root / '.cache' / 'hf_models'),
    resume_download=True,
)
print('Local model dir:', local_model_dir)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir, token=hf_token, legacy=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    token=hf_token,
    trust_remote_code=True,
    device_map='auto',
    torch_dtype=compute_dtype,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


In [ ]:
dataset = dataset.map(
    lambda row: {'text': (row.get('prompt') or '') + (row.get('completion') or '')},
    remove_columns=dataset['train'].column_names,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=gradient_accumulation_steps,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    bf16=supports_bf16,
    fp16=not supports_bf16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.model.print_trainable_parameters()


In [ ]:
trainer.train()


In [ ]:
trainer.save_model(str(OUTPUT_DIR / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'adapter'))


In [ ]:
test_queries = [
    'Покажи лимиты по Благовещенску по месяцам',
    'Покажи кассовые выплаты по 0502',
    'Покажи сумму контрактов по источнику gz',
]

def build_inference_prompt(query: str) -> str:
    return (
        'Ты преобразуешь русский запрос к системе бюджетной аналитики в JSON для API. '
        'Верни только JSON без пояснений.\n\n'
        f'Запрос пользователя:\n{query}\n\nВерни только JSON:'
    )

for query in test_queries:
    prompt = build_inference_prompt(query)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=220, temperature=0.0, do_sample=False)
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('QUERY:', query)
    print(generated)
    print('-' * 80)
